# PySpark — Reading & Writing Data

PySpark can read from and write to many formats. The pattern is always:

```python
# Read
df = spark.read.format("csv").option("k", "v").load("path")
# or shorthand
df = spark.read.csv("path", header=True, inferSchema=True)

# Write
df.write.format("parquet").mode("overwrite").save("path")
# or shorthand
df.write.parquet("path", mode="overwrite")
```

| Format | Best for | Size | Schema |
|--------|----------|------|--------|
| CSV | Human-readable exchange | Large | No |
| JSON | APIs, semi-structured data | Large | No |
| Parquet | Production analytics (columnar) | Small | Yes (embedded) |
| ORC | Hive/warehouse workloads | Small | Yes (embedded) |

**Use Parquet for everything in production** — it's columnar, compressed, and schema-aware.

## Setup

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("ReadingWriting") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

data = [
    (1, "Alice",   "Engineering", 95000, True),
    (2, "Bob",     "Marketing",   72000, False),
    (3, "Charlie", "Engineering", 110000, True),
    (4, "Diana",   "HR",          65000, True),
    (5, "Eve",     "Marketing",   85000, False),
]
schema = StructType([
    StructField("id",     IntegerType(), False),
    StructField("name",   StringType(),  False),
    StructField("dept",   StringType(),  True),
    StructField("salary", IntegerType(), True),
    StructField("active", BooleanType(), True),
])
df = spark.createDataFrame(data, schema)
df.show()

OUT = r"C:\Users\PS\Documents\Python-Exp\RawData\output"
os.makedirs(OUT, exist_ok=True)
print("Output dir:", OUT)

## 1. CSV — Read and Write

In [ ]:
# Write to CSV
csv_path = OUT + "\\employees_csv"
df.write.csv(csv_path, header=True, mode="overwrite")
print("Written to:", csv_path)

# Read back
df_csv = spark.read.csv(csv_path, header=True, inferSchema=True)
df_csv.show()
df_csv.printSchema()

# With explicit schema (preferred)
emp_schema = StructType([
    StructField("id",     IntegerType(), True),
    StructField("name",   StringType(),  True),
    StructField("dept",   StringType(),  True),
    StructField("salary", IntegerType(), True),
    StructField("active", BooleanType(), True),
])
df_csv_typed = spark.read.schema(emp_schema).option("header", "true").csv(csv_path)
df_csv_typed.printSchema()

## 2. Reading Multiple CSV Files

PySpark can read an entire directory of CSV files as a single DataFrame — very useful for daily/monthly partitioned files.

In [ ]:
# Read all CSV files in a directory
df_all = spark.read.csv(csv_path, header=True, inferSchema=True)
print("Rows from all CSVs in dir:", df_all.count())

# Glob pattern — read files matching a pattern
# Example: all monthly sales files
# df_monthly = spark.read.csv("s3://bucket/sales/2024-*.csv", header=True, inferSchema=True)

# Read specific files (list)
# df_multi = spark.read.csv(["file1.csv", "file2.csv"], header=True)

print("\nReading multiple files: just point to the directory or use a glob pattern.")
print("PySpark unions them automatically into one DataFrame.")

## 3. JSON — Read and Write

In [ ]:
# Write to JSON (one JSON object per line — JSON Lines format)
json_path = OUT + "\\employees_json"
df.write.json(json_path, mode="overwrite")
print("Written JSON to:", json_path)

# Read back
df_json = spark.read.json(json_path)
df_json.show()
df_json.printSchema()

# Nested JSON — PySpark handles it
nested_data = spark.read.json(spark.sparkContext.parallelize([
    '{"name": "Alice", "address": {"city": "NY", "zip": "10001"}, "skills": ["Python", "SQL"]}',
    '{"name": "Bob",   "address": {"city": "LA", "zip": "90001"}, "skills": ["Java", "Spark"]}',
]))
nested_data.printSchema()
nested_data.show(truncate=False)

## 4. Parquet — The Production Standard

Parquet is **columnar**, **compressed**, and **schema-embedded** — much faster for analytics than CSV.  
Always use Parquet for intermediate data and output in production pipelines.

In [ ]:
# Write to Parquet
parquet_path = OUT + "\\employees_parquet"
df.write.parquet(parquet_path, mode="overwrite")
print("Written Parquet to:", parquet_path)

# Read back — schema is embedded, no need to specify types
df_parquet = spark.read.parquet(parquet_path)
df_parquet.printSchema()   # types are preserved exactly
df_parquet.show()

# Parquet with compression (default is snappy)
df.write.option("compression", "snappy").parquet(parquet_path + "_snappy", mode="overwrite")

print("\nCSV vs Parquet:")
print("  CSV    — human readable, no schema, large file size")
print("  Parquet — binary, schema embedded, compressed, columnar = fast analytics")

## 5. Write Modes

| Mode | Behaviour |
|------|-----------|
| `overwrite` | Delete existing data and write fresh |
| `append` | Add new data to existing files |
| `ignore` | Do nothing if path already exists |
| `error` / `errorifexists` | Raise an error if path exists (default) |

In [ ]:
# overwrite — most common in ETL
df.write.parquet(parquet_path, mode="overwrite")

# append — daily incremental load
# df_today.write.parquet(parquet_path, mode="append")

# ignore — idempotent, safe to re-run
# df.write.parquet(parquet_path, mode="ignore")

print("Write modes: overwrite | append | ignore | error")

## 6. Partitioned Writes — Organise by Column Value

`partitionBy()` creates a folder structure based on column values — speeds up reads when filtering by that column.

In [ ]:
# Write partitioned by department
partitioned_path = OUT + "\\employees_by_dept"
df.write.partitionBy("dept").parquet(partitioned_path, mode="overwrite")

# Directory structure created:
# employees_by_dept/
#   dept=Engineering/   ← part files for Engineering only
#   dept=HR/
#   dept=Marketing/

# Read a specific partition — only that folder is scanned (predicate pushdown)
df_eng = spark.read.parquet(partitioned_path).filter("dept = 'Engineering'")
df_eng.show()
print("Partitioned write enables faster reads by partition column.")

## 7. Controlling Output File Count — coalesce() and repartition()

In [ ]:
# By default, PySpark writes one file per partition (often 200 files for large data)
# coalesce(1) — merge into a single output file (small data / reporting only)
single_path = OUT + "\\employees_single"
df.coalesce(1).write.csv(single_path, header=True, mode="overwrite")
print("Single file output written to:", single_path)

# WARNING: coalesce(1) moves all data to one executor — only for small datasets
print("\ncoalesce(1) → 1 file, all data moved to one node — OK for small datasets only")
print("repartition(N) → exactly N files, full shuffle — use for large datasets")

## Quick Reference

```python
# CSV
df = spark.read.csv("path", header=True, inferSchema=True)
df = spark.read.schema(schema).option("header", True).csv("path")
df.write.csv("path", header=True, mode="overwrite")

# JSON
df = spark.read.json("path")
df.write.json("path", mode="overwrite")

# Parquet (preferred for production)
df = spark.read.parquet("path")
df.write.parquet("path", mode="overwrite")

# Write modes: overwrite | append | ignore | error
df.write.mode("overwrite").parquet("path")

# Partitioned write
df.write.partitionBy("year", "month").parquet("path")

# Single file output
df.coalesce(1).write.csv("path", header=True)
```

## Interview Questions

1. What is the difference between CSV and Parquet? When would you use each?
2. What is `inferSchema` and what is the cost of using it?
3. What are the four write modes in PySpark?
4. What does `partitionBy()` do and why does it speed up reads?
5. How do you read all CSV files in a directory with PySpark?
6. Why is `coalesce(1)` dangerous for large datasets?
7. What is the default number of output partitions when writing a PySpark DataFrame?

## Practice Exercises

**Easy:**
1. Write `df` to CSV with a header. Read it back and verify the row count.
2. Write `df` to Parquet. Read it back — does `printSchema()` show the correct types?

**Medium:**
3. Write `df` partitioned by `dept`. Then read only the Engineering partition.
4. Read a JSON file and flatten any nested fields using `select()` and `col("nested.field")`.

**Advanced:**
5. Read the Cabs.csv file with an explicit schema (not inferSchema). Compare load time vs inferSchema.
6. Build a pipeline that reads CSV, cleans nulls, and writes Parquet partitioned by `active` status.

In [ ]:
spark.stop()